# Ceramic Strength

**NIST/SEMATECH e-Handbook of Statistical Methods, Section 1.4.2.10**

Source: [https://www.itl.nist.gov/div898/handbook/eda/section4/eda42a1.htm](https://www.itl.nist.gov/div898/handbook/eda/section4/eda42a1.htm)

## Background

Said Jahanmir, NIST Ceramics Division, ceramic flexural strength (1996).

This is a **2^4 full factorial design of experiment (DOE)** with factors:
- **X1:** Table Speed (-1 = slow, +1 = fast)
- **X2:** Down Feed Rate (-1 = slow, +1 = fast)
- **X3:** Wheel Grit (-1 = 140/170, +1 = 80/100)
- **X4:** Direction (-1 = longitudinal, +1 = transverse)

Only **longitudinal data (X4 = -1)** are analyzed here (480 of 960 total observations).

### Analysis Goals

1. **Factor Rankings:** Determine the strongest factor affecting ceramic strength
2. **Effect Magnitudes:** Estimate the effect size of each factor
3. **Optimal Settings:** Determine optimal factor settings for maximum strength
4. **Batch Variability:** Assess batch-to-batch variability as a nuisance factor

## Environment Setup

In [ ]:
# Check dependencies and install if missing
try:
    import numpy as np
    import scipy
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns
except ImportError:
    !pip install numpy scipy pandas matplotlib seaborn
    import numpy as np
    import scipy
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns

print(f'NumPy {np.__version__}, SciPy {scipy.__version__}, Pandas {pd.__version__}')
print(f'Matplotlib {plt.matplotlib.__version__}, Seaborn {sns.__version__}')

In [ ]:
# Quantum Explorer dark theme for matplotlib/seaborn
# Matches the site color scheme for visual consistency

import matplotlib.pyplot as plt
import seaborn as sns

QUANTUM_COLORS = {
    'background': '#0f1117',
    'surface': '#1a1d27',
    'accent': '#e06040',
    'teal': '#00a3a3',
    'text': '#e8e8f0',
    'text_secondary': '#9ca3af',
    'border': '#2a2d3a',
    'gradient_start': '#e06040',
    'gradient_end': '#00a3a3',
}

# Color cycle for multiple series
QUANTUM_PALETTE = ['#e06040', '#00a3a3', '#f0c040', '#a080e0', '#60c0a0', '#e080a0']

plt.rcParams.update({
    'figure.facecolor': QUANTUM_COLORS['background'],
    'axes.facecolor': QUANTUM_COLORS['surface'],
    'axes.edgecolor': QUANTUM_COLORS['border'],
    'axes.labelcolor': QUANTUM_COLORS['text'],
    'axes.titlecolor': QUANTUM_COLORS['text'],
    'xtick.color': QUANTUM_COLORS['text_secondary'],
    'ytick.color': QUANTUM_COLORS['text_secondary'],
    'text.color': QUANTUM_COLORS['text'],
    'grid.color': QUANTUM_COLORS['border'],
    'grid.alpha': 0.5,
    'figure.figsize': [10, 6],
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
    'axes.prop_cycle': plt.cycler('color', QUANTUM_PALETTE),
})

sns.set_theme(style='darkgrid', rc={
    'axes.facecolor': QUANTUM_COLORS['surface'],
    'figure.facecolor': QUANTUM_COLORS['background'],
    'grid.color': QUANTUM_COLORS['border'],
    'text.color': QUANTUM_COLORS['text'],
    'axes.labelcolor': QUANTUM_COLORS['text'],
    'xtick.color': QUANTUM_COLORS['text_secondary'],
    'ytick.color': QUANTUM_COLORS['text_secondary'],
})

print('Quantum Explorer theme configured.')

In [ ]:
# Additional imports for statistical analysis
from scipy import stats
from io import StringIO
import warnings
warnings.filterwarnings('ignore')

## Data Loading

Load the **JAHANMI2.DAT** dataset from NIST (480 observations).

In [ ]:
# Load dataset (CSV generated from NIST JAHANMI2.DAT)
DATA_FILE = 'ceramic-strength.csv'
GITHUB_URL = 'https://raw.githubusercontent.com/PatrykQuantumNomad/PatrykQuantumNomad/main/notebooks/eda/data/ceramic-strength.csv'

try:
    df = pd.read_csv(DATA_FILE)
except FileNotFoundError:
    df = pd.read_csv(GITHUB_URL)

print(f'Loaded {len(df)} rows')
assert len(df) == 480, f'Expected 480 rows, got {len(df)}'

In [ ]:
# Preview the first few rows
print(df.head(10))
print()
print(f'Shape: {df.shape}')
print(f'Data types:\n{df.dtypes}')

## Summary Statistics

Compute key descriptive statistics for the **Y** variable.

In [ ]:
# Summary statistics for Y
y = df['Y']

summary = pd.DataFrame({
    'Statistic': ['Mean', 'Std Dev', 'Median', 'Min', 'Max',
                  'Skewness', 'Kurtosis', 'N'],
    'Value': [
        y.mean(),
        y.std(ddof=1),
        y.median(),
        y.min(),
        y.max(),
        y.skew(),
        y.kurtosis(),
        len(y),
    ]
})

print(summary.to_string(index=False))

## 4-Plot Analysis

The 4-plot is a collection of four graphical EDA techniques whose purpose is to test 
the assumptions that underlie most measurement processes:

1. **Run Sequence Plot** (upper left) -- tests fixed location and variation
2. **Lag Plot** (upper right) -- tests randomness
3. **Histogram** (lower left) -- tests distributional assumptions
4. **Normal Probability Plot** (lower right) -- tests normality

In [ ]:
# 4-Plot: 4-Plot of Ceramic Strength
y = df['Y'].values

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('4-Plot of Ceramic Strength',
             fontsize=16, color=QUANTUM_COLORS['text'], y=0.98)

# 1. Run Sequence Plot (upper left)
ax = axes[0, 0]
ax.plot(range(len(y)), y, '.', color=QUANTUM_COLORS['accent'],
        markersize=3, alpha=0.7)
ax.axhline(y.mean(), color=QUANTUM_COLORS['teal'],
           linestyle='--', linewidth=1, label=f'Mean = {y.mean():.4f}')
ax.set_xlabel('Observation Number')
ax.set_ylabel('Y')
ax.set_title('Run Sequence Plot')
ax.legend(fontsize=9)

# 2. Lag Plot (upper right)
ax = axes[0, 1]
ax.scatter(y[:-1], y[1:], c=QUANTUM_COLORS['accent'],
           s=5, alpha=0.5)
ax.set_xlabel('Y(i)')
ax.set_ylabel('Y(i+1)')
ax.set_title('Lag Plot (lag=1)')

# 3. Histogram (lower left)
ax = axes[1, 0]
ax.hist(y, bins='auto', color=QUANTUM_COLORS['accent'],
        edgecolor=QUANTUM_COLORS['border'], alpha=0.8)
ax.set_xlabel('Y')
ax.set_ylabel('Frequency')
ax.set_title('Histogram')

# 4. Normal Probability Plot (lower right)
ax = axes[1, 1]
stats.probplot(y, dist='norm', plot=ax)
ax.get_lines()[0].set_color(QUANTUM_COLORS['accent'])
ax.get_lines()[1].set_color(QUANTUM_COLORS['teal'])
ax.set_title('Normal Probability Plot')

plt.tight_layout()
plt.show()

## Batch Effect Analysis

The 4-plot histogram reveals a **bimodal distribution**, suggesting a dominant batch effect.
Before analyzing primary factors (X1, X2, X3), we must first quantify the batch-to-batch
variability. This is standard practice in DOE: nuisance factors are investigated before
primary factors.

In [ ]:
# Split data by Batch
batch1 = df[df['Batch'] == 1]['Y']
batch2 = df[df['Batch'] == 2]['Y']

print(f'Batch 1: N={len(batch1)}, Mean={batch1.mean():.4f}, SD={batch1.std(ddof=1):.4f}')
print(f'Batch 2: N={len(batch2)}, Mean={batch2.mean():.4f}, SD={batch2.std(ddof=1):.4f}')
print(f'Batch effect (mean difference): {batch1.mean() - batch2.mean():.4f}')

In [ ]:
# Bihistogram: Batch 1 vs Batch 2
fig, ax = plt.subplots(figsize=(10, 6))

ax.hist(batch1, bins=25, alpha=0.6, color=QUANTUM_COLORS['accent'],
        label=f'Batch 1 (mean={batch1.mean():.2f})', edgecolor='none')
ax.hist(batch2, bins=25, alpha=0.6, color=QUANTUM_COLORS['teal'],
        label=f'Batch 2 (mean={batch2.mean():.2f})', edgecolor='none')

ax.set_xlabel('Ceramic Strength (Y)')
ax.set_ylabel('Frequency')
ax.set_title('Bihistogram: Batch 1 vs Batch 2', color=QUANTUM_COLORS['text'])
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Side-by-side box plots by Batch
fig, ax = plt.subplots(figsize=(8, 6))

sns.boxplot(data=df, x='Batch', y='Y', ax=ax,
            palette=[QUANTUM_COLORS['accent'], QUANTUM_COLORS['teal']])
ax.set_title('Ceramic Strength by Batch', color=QUANTUM_COLORS['text'])
ax.set_xlabel('Batch')
ax.set_ylabel('Strength (Y)')
plt.tight_layout()
plt.show()

In [ ]:
# Two-sample t-test for batch means
from scipy.stats import ttest_ind, f_oneway

t_stat, t_pval = ttest_ind(batch1, batch2)
print(f'Two-sample t-test: T = {t_stat:.4f}, p-value = {t_pval:.2e}')

# Pooled standard deviation
n1, n2 = len(batch1), len(batch2)
s1, s2 = batch1.std(ddof=1), batch2.std(ddof=1)
pooled_sd = np.sqrt(((n1-1)*s1**2 + (n2-1)*s2**2) / (n1+n2-2))
print(f'Pooled SD = {pooled_sd:.4f}')

# F-test for equal variances
f_stat = s1**2 / s2**2
print(f'F-test for equal variances: F = {f_stat:.3f}')

### NIST Reference Comparison

| Statistic | Batch 1 | Batch 2 |
|-----------|---------|---------|
| N | 240 | 240 |
| Mean | 688.9987 | 611.1559 |
| SD | 65.5491 | 61.8543 |

**T-statistic:** 13.3806 (two-sample t-test for equal means)

The batch effect is approximately **75-100 units**, confirming that batch is a dominant
nuisance factor. All subsequent factor analysis should be performed **within each batch**.

## Lab Effect Analysis

Before analyzing primary factors, we also check for **lab effects** (nuisance factor).
The data were collected across multiple labs, and systematic lab differences could
confound factor effects.

In [ ]:
# Box plots by Lab
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Overall by lab
ax = axes[0]
sns.boxplot(data=df, x='Lab', y='Y', ax=ax,
            color=QUANTUM_COLORS['accent'])
ax.set_title('Strength by Lab (Overall)', color=QUANTUM_COLORS['text'])
ax.set_xlabel('Lab')
ax.set_ylabel('Strength (Y)')

# By lab within each batch
ax = axes[1]
for batch_num, color in zip([1, 2], [QUANTUM_COLORS['accent'], QUANTUM_COLORS['teal']]):
    batch_data = df[df['Batch'] == batch_num]
    lab_means = batch_data.groupby('Lab')['Y'].mean()
    ax.plot(lab_means.index, lab_means.values, 'o-',
            color=color, label=f'Batch {batch_num}', linewidth=2, markersize=6)
ax.set_title('Mean Strength by Lab within Batch', color=QUANTUM_COLORS['text'])
ax.set_xlabel('Lab')
ax.set_ylabel('Mean Strength')
ax.legend()

plt.tight_layout()
plt.show()

## Primary Factor Analysis

DOE mean plots show the average response at each factor level. The factor with the
largest difference between its high (+1) and low (-1) level means has the greatest
effect on the response. We compute these **per batch** since the batch effect dominates.

In [ ]:
# DOE Mean Plots: average Y at each factor level, per batch
factors = ['X1', 'X2', 'X3']
factor_names = {'X1': 'Table Speed', 'X2': 'Down Feed Rate', 'X3': 'Wheel Grit'}

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('DOE Mean Plots by Factor and Batch',
             fontsize=16, color=QUANTUM_COLORS['text'], y=1.02)

for batch_num, ax_row in zip([1, 2], axes):
    batch_data = df[df['Batch'] == batch_num]
    for i, factor in enumerate(factors):
        means = batch_data.groupby(factor)['Y'].mean()
        ax = ax_row[i]
        ax.plot(means.index, means.values, 'o-',
                color=QUANTUM_COLORS['accent'], linewidth=2, markersize=8)
        effect = means.values[-1] - means.values[0]
        ax.set_title(f'{factor_names[factor]}\nBatch {batch_num} (effect={effect:.2f})',
                     fontsize=11)
        ax.set_xlabel(f'{factor} Level')
        ax.set_ylabel('Mean Strength')
        ax.set_xticks([-1, 1])

plt.tight_layout()
plt.show()

In [ ]:
# DOE SD Plots: standard deviation of Y at each factor level, per batch
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('DOE SD Plots by Factor and Batch',
             fontsize=16, color=QUANTUM_COLORS['text'], y=1.02)

for batch_num, ax_row in zip([1, 2], axes):
    batch_data = df[df['Batch'] == batch_num]
    for i, factor in enumerate(factors):
        sds = batch_data.groupby(factor)['Y'].std()
        ax = ax_row[i]
        ax.plot(sds.index, sds.values, 's-',
                color=QUANTUM_COLORS['teal'], linewidth=2, markersize=8)
        ax.set_title(f'{factor_names[factor]} - Batch {batch_num}', fontsize=11)
        ax.set_xlabel(f'{factor} Level')
        ax.set_ylabel('SD of Strength')
        ax.set_xticks([-1, 1])

plt.tight_layout()
plt.show()

### Factor Rankings by Batch

Factor rankings **differ by batch**, a key finding of this analysis:

| Rank | Batch 1 | Effect | Batch 2 | Effect |
|------|---------|--------|---------|--------|
| 1 | X1 (Table Speed) | -30.77 | X2 (Down Feed Rate) | 18.22 |
| 2 | X2 (Down Feed Rate) | 12.48 | X1 (Table Speed) | -14.98 |
| 3 | X3 (Wheel Grit) | 5.07 | X3 (Wheel Grit) | 6.14 |

In **Batch 1**, X1 (table speed) is the dominant factor with effect = -30.77.
In **Batch 2**, X2 (down feed rate) is the dominant factor with effect = 18.22.
This reversal of factor importance across batches is critical for optimization.

In [ ]:
# Interaction Effects: mean response for each factor combination, per batch
factor_pairs = [('X1', 'X2'), ('X1', 'X3'), ('X2', 'X3')]

fig, axes = plt.subplots(2, 3, figsize=(15, 10))
fig.suptitle('Interaction Plots by Batch',
             fontsize=16, color=QUANTUM_COLORS['text'], y=1.02)

for batch_num, ax_row in zip([1, 2], axes):
    batch_data = df[df['Batch'] == batch_num]
    for i, (f1, f2) in enumerate(factor_pairs):
        interaction = batch_data.groupby([f1, f2])['Y'].mean().unstack()
        ax = ax_row[i]
        for col in interaction.columns:
            ax.plot(interaction.index, interaction[col], 'o-',
                    label=f'{f2}={col}', linewidth=2, markersize=6)
        ax.set_title(f'{f1} x {f2} - Batch {batch_num}', fontsize=11)
        ax.set_xlabel(f'{f1} Level')
        ax.set_ylabel('Mean Strength')
        ax.set_xticks([-1, 1])
        ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

### Interaction Effects Interpretation

Non-parallel lines in the interaction plots indicate **interaction effects** between factors.
When two lines cross or diverge substantially, the effect of one factor depends on the level
of the other factor. Key observations:

- **X1 x X2:** Moderate interaction in both batches
- **X1 x X3:** Weak interaction
- **X2 x X3:** Weak interaction

The presence of interactions means that factor effects cannot be interpreted in isolation.
Optimal settings must consider factor combinations, not just individual factor levels.

## One-Way ANOVA

One-way ANOVA tests whether the mean response differs significantly across
levels of each factor. We perform this **per batch** since the batch effect dominates.
We use `scipy.stats.f_oneway` which returns the F-statistic and p-value.

In [ ]:
# One-way ANOVA for each factor, per batch
from scipy.stats import f_oneway

anova_results = []

for batch_num in [1, 2]:
    batch_data = df[df['Batch'] == batch_num]
    for factor in ['X1', 'X2', 'X3']:
        groups = [group['Y'].values for _, group in batch_data.groupby(factor)]
        f_stat, p_val = f_oneway(*groups)
        anova_results.append({
            'Factor': factor,
            'Batch': batch_num,
            'F-statistic': f_stat,
            'p-value': p_val,
            'Significant': 'Yes' if p_val < 0.05 else 'No'
        })

anova_df = pd.DataFrame(anova_results)
print(anova_df.to_string(index=False))

In [ ]:
# ANOVA summary visualization
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for i, batch_num in enumerate([1, 2]):
    batch_results = anova_df[anova_df['Batch'] == batch_num]
    ax = axes[i]
    colors = [QUANTUM_COLORS['accent'] if sig == 'Yes' else QUANTUM_COLORS['border']
             for sig in batch_results['Significant']]
    ax.barh(batch_results['Factor'], batch_results['F-statistic'], color=colors)
    ax.set_title(f'ANOVA F-statistics - Batch {batch_num}',
                 color=QUANTUM_COLORS['text'])
    ax.set_xlabel('F-statistic')
    ax.axvline(x=3.84, color=QUANTUM_COLORS['teal'], linestyle='--',
              label='F critical (approx)', linewidth=1)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

### ANOVA Interpretation

The ANOVA results confirm the factor ranking findings from the DOE mean plots:

- **Batch 1:** X1 (Table Speed) shows the largest F-statistic, confirming it as the
  dominant factor. X2 and X3 are also significant but with smaller effects.
- **Batch 2:** X2 (Down Feed Rate) shows the largest F-statistic, confirming it as the
  dominant factor in this batch. X1 remains significant but less dominant.

The differing factor significance across batches reinforces that the batch effect
fundamentally changes the factor-response relationship.

## Conclusions

### Key Findings

1. **Batch Effect Dominates:** The batch-to-batch difference (~75 units) is the
   largest single source of variation. Batch 1 mean = 688.9987, Batch 2 mean = 611.1559
   (T = 13.3806, highly significant).

2. **Factor Rankings Differ by Batch:**
   - **Batch 1:** X1 (Table Speed) is dominant (effect = -30.77)
   - **Batch 2:** X2 (Down Feed Rate) is dominant (effect = 18.22)
   - X3 (Wheel Grit) ranks third in both batches

3. **Interaction Effects Present:** Non-parallel interaction plots indicate that
   factor effects are not purely additive. The X1-X2 interaction is most notable.

4. **Practical Implications:**
   - The dominant batch effect must be controlled before optimizing factor settings
   - Optimal factor settings depend on which batch is being produced
   - A single set of optimal settings does not exist across batches

### NIST Reference

Full analysis: [https://www.itl.nist.gov/div898/handbook/eda/section4/eda42a1.htm](https://www.itl.nist.gov/div898/handbook/eda/section4/eda42a1.htm)